In [1]:
# Install Required Libraries
!pip -q install pandas==2.2.2
!pip -q install -U datasets transformers accelerate sentencepiece evaluate tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.3 MB/s eta 0:00:00


In [2]:

# Import Libraries
import json
import pandas as pd

from datasets import load_dataset
from tqdm import tqdm

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
import datasets
import transformers

print("Datasets Version:", datasets.__version__)
print("Transformers Version:", transformers.__version__)

Datasets Version: 5.0.0
Transformers Version: 5.14.1


In [4]:

# Load Empathetic Dialogues


empathetic = load_dataset(
    "pixelsandpointers/empathetic_dialogues_for_lm"
)

print(empathetic)

dataset_infos.json:   0%|          | 0.00/886 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.53MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  801kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/17844 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2763 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2542 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['conv'],
        num_rows: 17844
    })
    validation: Dataset({
        features: ['conv'],
        num_rows: 2763
    })
    test: Dataset({
        features: ['conv'],
        num_rows: 2542
    })
})


In [5]:
print(empathetic["train"][0])

{'conv': ['I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.', 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.', 'Was this a friend you were in love with_comma_ or just a best friend?', 'This was a best friend. I miss her.', 'Where has she gone?', 'We no longer talk.', 'Oh was this something that happened because of an argument?']}


In [6]:
print(empathetic["train"].column_names)

['conv']


In [6]:
print(empathetic["train"][0]["conv"])

['I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.', 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.', 'Was this a friend you were in love with_comma_ or just a best friend?', 'This was a best friend. I miss her.', 'Where has she gone?', 'We no longer talk.', 'Oh was this something that happened because of an argument?']


In [7]:
import re

def clean_text(text):
    """
    Clean placeholder tokens and normalize whitespace.
    """

    if not isinstance(text, str):
        return ""

    replacements = {
        "_comma_": ",",
        "_period_": ".",
        "_question_": "?",
        "_exclamation_": "!",
        "_semicolon_": ";",
        "_colon_": ":",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [8]:
sample = "There were many people_comma_ but we felt alone_period_"

print(clean_text(sample))

There were many people, but we felt alone.


In [9]:
conversation_data = []

for sample in empathetic["train"]:

    conversation = sample["conv"]

    # Skip conversations with fewer than 2 turns
    if len(conversation) < 2:
        continue

    for i in range(len(conversation) - 1):

        user_input = clean_text(conversation[i])
        assistant_response = clean_text(conversation[i + 1])

        conversation_data.append({
            "instruction": "Respond naturally, warmly, and empathetically to the user.",
            "input": user_input,
            "response": assistant_response,
            "category": "conversation",
            "source": "EmpatheticDialogues"
        })

In [10]:
print("Total training samples:", len(conversation_data))

Total training samples: 76673


In [11]:
for i in range(3):
    print(conversation_data[i])
    print("-" * 80)

{'instruction': 'Respond naturally, warmly, and empathetically to the user.', 'input': 'I remember going to the fireworks with my best friend. There was a lot of people, but it only felt like us in the world.', 'response': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.', 'category': 'conversation', 'source': 'EmpatheticDialogues'}
--------------------------------------------------------------------------------
{'instruction': 'Respond naturally, warmly, and empathetically to the user.', 'input': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.', 'response': 'Was this a friend you were in love with, or just a best friend?', 'category': 'conversation', 'source': 'EmpatheticDialogues'}
------------

In [12]:
from datasets import load_dataset

persona = load_dataset("AlekseyKorshuk/persona-chat")

dataset_infos.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 97.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.82MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/17878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
print(persona)

DatasetDict({
    train: Dataset({
        features: ['personality', 'utterances'],
        num_rows: 17878
    })
    validation: Dataset({
        features: ['personality', 'utterances'],
        num_rows: 1000
    })
})


In [14]:
print(persona["train"].column_names)

['personality', 'utterances']


In [15]:
print(persona["train"][0])

{'personality': ['i like to remodel homes .', 'i like to go hunting .', 'i like to shoot a bow .', 'my favorite holiday is halloween .'], 'utterances': [{'candidates': ['my mom was single with 3 boys , so we never left the projects .', 'i try to wear all black every day . it makes me feel comfortable .', 'well nursing stresses you out so i wish luck with sister', 'yeah just want to pick up nba nfl getting old', 'i really like celine dion . what about you ?', 'no . i live near farms .', "i wish i had a daughter , i'm a boy mom . they're beautiful boys though still lucky", 'yeah when i get bored i play gone with the wind my favorite movie .', "hi how are you ? i'm eating dinner with my hubby and 2 kids .", 'were you married to your high school sweetheart ? i was .', 'that is great to hear ! are you a competitive rider ?', "hi , i'm doing ok . i'm a banker . how about you ?", "i'm 5 years old", 'hi there . how are you today ?', 'i totally understand how stressful that can be .', 'yeah som

In [16]:
print(persona["train"][0]["utterances"][0])

{'candidates': ['my mom was single with 3 boys , so we never left the projects .', 'i try to wear all black every day . it makes me feel comfortable .', 'well nursing stresses you out so i wish luck with sister', 'yeah just want to pick up nba nfl getting old', 'i really like celine dion . what about you ?', 'no . i live near farms .', "i wish i had a daughter , i'm a boy mom . they're beautiful boys though still lucky", 'yeah when i get bored i play gone with the wind my favorite movie .', "hi how are you ? i'm eating dinner with my hubby and 2 kids .", 'were you married to your high school sweetheart ? i was .', 'that is great to hear ! are you a competitive rider ?', "hi , i'm doing ok . i'm a banker . how about you ?", "i'm 5 years old", 'hi there . how are you today ?', 'i totally understand how stressful that can be .', 'yeah sometimes you do not know what you are actually watching', 'mother taught me to cook ! we are looking for an exterminator .', 'i enjoy romantic movie . what

In [17]:
print(persona["train"][0]["utterances"][0].keys())

dict_keys(['candidates', 'history'])


In [18]:
sample = persona["train"][0]["utterances"][0]

print("History:")
print(sample["history"])

print("\nNumber of candidates:")
print(len(sample["candidates"]))

print("\nLast candidate:")
print(sample["candidates"][-1])

History:
["hi , how are you doing ? i'm getting ready to do some cheetah chasing to stay in shape ."]

Number of candidates:
20

Last candidate:
you must be very fast . hunting is one of my favorite hobbies .


In [19]:
# preprocessing personalchat
persona_records = []

for sample in persona["train"]:

    personality = sample["personality"]

    for dialog in sample["utterances"]:

        history = dialog["history"]

        candidates = dialog["candidates"]

        # Skip incomplete conversations
        if len(history) == 0:
            continue

        # Last message from the conversation
        user_input = clean_text(history[-1])

        # Correct response (last candidate)
        assistant_response = clean_text(candidates[-1])

        # Skip very short responses
        if len(user_input) < 3 or len(assistant_response) < 3:
            continue

        persona_records.append({

            "instruction": "Continue the conversation in a friendly, engaging, and natural way.",

            "input": user_input,

            "response": assistant_response,

            "personality": personality,

            "category": "conversation",

            "source": "PersonaChat"

        })

print(f"Total PersonaChat samples: {len(persona_records):,}")

Total PersonaChat samples: 131,345


In [20]:
for i in range(5):
    print(persona_records[i])
    print("=" * 80)

{'instruction': 'Continue the conversation in a friendly, engaging, and natural way.', 'input': "hi , how are you doing ? i'm getting ready to do some cheetah chasing to stay in shape .", 'response': 'you must be very fast . hunting is one of my favorite hobbies .', 'personality': ['i like to remodel homes .', 'i like to go hunting .', 'i like to shoot a bow .', 'my favorite holiday is halloween .'], 'category': 'conversation', 'source': 'PersonaChat'}
{'instruction': 'Continue the conversation in a friendly, engaging, and natural way.', 'input': 'i am ! for my hobby i like to do canning or some whittling .', 'response': 'i also remodel homes when i am not out bow hunting .', 'personality': ['i like to remodel homes .', 'i like to go hunting .', 'i like to shoot a bow .', 'my favorite holiday is halloween .'], 'category': 'conversation', 'source': 'PersonaChat'}
{'instruction': 'Continue the conversation in a friendly, engaging, and natural way.', 'input': "that's neat . when i was in 

In [21]:
import json

with open("persona_train.json", "w", encoding="utf-8") as f:
    json.dump(persona_records, f, indent=4, ensure_ascii=False)

print("PersonaChat saved successfully.")

PersonaChat saved successfully.


In [22]:
import json

with open("empathetic_train.json", "w", encoding="utf-8") as f:
    json.dump(conversation_data, f, indent=4, ensure_ascii=False)

print(f"EmpatheticDialogues saved successfully!")
print(f"Total samples: {len(conversation_data):,}")

EmpatheticDialogues saved successfully!
Total samples: 76,673


In [28]:
from datasets import load_dataset

daily = load_dataset("elricwan/dailydialog")

README.md:   0%|          | 0.00/348 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.06MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/13118 [00:00<?, ? examples/s]

In [29]:
print(daily)

DatasetDict({
    train: Dataset({
        features: ['conversation'],
        num_rows: 13118
    })
})


In [30]:
print(daily["train"].column_names)

['conversation']


In [31]:
print(daily["train"][0])

{'conversation': ['Person A: The kitchen stinks .', "Person B: I'll throw out the garbage ."]}


In [32]:
daily_records = []

for sample in daily["train"]:

    conversation = sample["conversation"]

    # Skip conversations with fewer than 2 turns
    if len(conversation) < 2:
        continue

    # Create input-response pairs
    for i in range(len(conversation) - 1):

        user_input = clean_text(conversation[i])
        assistant_response = clean_text(conversation[i + 1])

        if len(user_input) < 3 or len(assistant_response) < 3:
            continue

        daily_records.append({

            "instruction": "Respond naturally and appropriately in the conversation.",

            "input": user_input,

            "response": assistant_response,

            "category": "conversation",

            "source": "DailyDialog"

        })

print(f"Total DailyDialog samples: {len(daily_records):,}")

Total DailyDialog samples: 89,862


In [33]:
for i in range(5):
    print(daily_records[i])
    print("=" * 80)

{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': 'Person A: The kitchen stinks .', 'response': "Person B: I'll throw out the garbage .", 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': 'Person A: So Dick , how about getting some coffee for tonight ?', 'response': "Person B: Coffee ? I don ' t honestly like that kind of stuff .", 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': "Person B: Coffee ? I don ' t honestly like that kind of stuff .", 'response': 'Person A: Come on , you can at least try a little , besides your cigarette .', 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': 'Person A: Come on , you can at least try a little , besides your cigarette .', 'response': "Person B: What 

In [34]:
import json

with open("dailydialog_train.json", "w", encoding="utf-8") as f:
    json.dump(daily_records, f, indent=4, ensure_ascii=False)

print("DailyDialog saved successfully!")
print(f"Total samples: {len(daily_records):,}")

DailyDialog saved successfully!
Total samples: 89,862


In [35]:
user_input = clean_text(conversation[i])
assistant_response = clean_text(conversation[i + 1])

# Remove speaker labels
user_input = user_input.replace("Person A:", "").replace("Person B:", "").strip()
assistant_response = assistant_response.replace("Person A:", "").replace("Person B:", "").strip()

In [37]:
import json

# Load the saved file
with open("dailydialog_train.json", "r", encoding="utf-8") as f:
    daily_records = json.load(f)

# Remove speaker labels
for record in daily_records:
    record["input"] = (
        record["input"]
        .replace("Person A:", "")
        .replace("Person B:", "")
        .strip()
    )

    record["response"] = (
        record["response"]
        .replace("Person A:", "")
        .replace("Person B:", "")
        .strip()
    )

# Save the cleaned file
with open("dailydialog_train.json", "w", encoding="utf-8") as f:
    json.dump(daily_records, f, indent=4, ensure_ascii=False)

print(f"✅ Cleaned and saved {len(daily_records):,} DailyDialog records.")

✅ Cleaned and saved 89,862 DailyDialog records.


In [38]:
for i in range(5):
    print(daily_records[i])
    print("=" * 80)

{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': 'The kitchen stinks .', 'response': "I'll throw out the garbage .", 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': 'So Dick , how about getting some coffee for tonight ?', 'response': "Coffee ? I don ' t honestly like that kind of stuff .", 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': "Coffee ? I don ' t honestly like that kind of stuff .", 'response': 'Come on , you can at least try a little , besides your cigarette .', 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': 'Come on , you can at least try a little , besides your cigarette .', 'response': "What ' s wrong with that ? Cigarette is the thing I go crazy for .", 'category': 'con

In [39]:
import json

with open("dailydialog_train.json", "w", encoding="utf-8") as f:
    json.dump(daily_records, f, indent=4, ensure_ascii=False)

print("✅ DailyDialog saved successfully.")

✅ DailyDialog saved successfully.


In [40]:
for i in range(3):
    print(daily_records[i])
    print("=" * 80)

{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': 'The kitchen stinks .', 'response': "I'll throw out the garbage .", 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': 'So Dick , how about getting some coffee for tonight ?', 'response': "Coffee ? I don ' t honestly like that kind of stuff .", 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': "Coffee ? I don ' t honestly like that kind of stuff .", 'response': 'Come on , you can at least try a little , besides your cigarette .', 'category': 'conversation', 'source': 'DailyDialog'}


In [41]:
import os

os.makedirs("Conversation", exist_ok=True)

print("Folder created!")

Folder created!


In [42]:
import shutil

shutil.move("empathetic_train.json", "Conversation/empathetic_train.json")
shutil.move("persona_train.json", "Conversation/persona_train.json")
shutil.move("dailydialog_train.json", "Conversation/dailydialog_train.json")

print("All files moved successfully!")

All files moved successfully!


In [43]:
import os

print(os.listdir("Conversation"))

['dailydialog_train.json', 'empathetic_train.json', 'persona_train.json']


In [44]:
import json

with open("Conversation/empathetic_train.json", "r", encoding="utf-8") as f:
    empathetic = json.load(f)

with open("Conversation/persona_train.json", "r", encoding="utf-8") as f:
    persona = json.load(f)

with open("Conversation/dailydialog_train.json", "r", encoding="utf-8") as f:
    daily = json.load(f)

print(f"EmpatheticDialogues : {len(empathetic):,}")
print(f"PersonaChat         : {len(persona):,}")
print(f"DailyDialog         : {len(daily):,}")

EmpatheticDialogues : 76,673
PersonaChat         : 131,345
DailyDialog         : 89,862


In [45]:
conversation_train = empathetic + persona + daily

print(f"Before removing duplicates: {len(conversation_train):,}")

Before removing duplicates: 297,880


In [46]:
unique_records = []
seen = set()

for record in conversation_train:

    key = (
        record["input"].strip().lower(),
        record["response"].strip().lower()
    )

    if key not in seen:
        seen.add(key)
        unique_records.append(record)

conversation_train = unique_records

print(f"After removing duplicates: {len(conversation_train):,}")

After removing duplicates: 283,268


In [47]:
import random

random.seed(42)

random.shuffle(conversation_train)

In [48]:
with open("Conversation/conversation_train.json", "w", encoding="utf-8") as f:
    json.dump(
        conversation_train,
        f,
        indent=4,
        ensure_ascii=False
    )

print("✅ conversation_train.json saved successfully!")

✅ conversation_train.json saved successfully!


In [49]:
print(f"Total training samples: {len(conversation_train):,}")

for i in range(5):
    print(conversation_train[i])
    print("=" * 80)

Total training samples: 283,268
{'instruction': 'Continue the conversation in a friendly, engaging, and natural way.', 'input': 'hey my dog , george goes with to the beach . i can sell anything lol', 'response': "that's cool . have you traveled outside of the us ?", 'personality': ['i am a dog walker .', 'i have never traveled outside of the united states .', 'i live in new york .', 'my best friend lives in japan .', 'i eat ice cream when i am sad .'], 'category': 'conversation', 'source': 'PersonaChat'}
{'instruction': 'Respond naturally, warmly, and empathetically to the user.', 'input': 'So excited for this weekend!', 'response': 'Then why are you excite about it?', 'category': 'conversation', 'source': 'EmpatheticDialogues'}
{'instruction': 'Continue the conversation in a friendly, engaging, and natural way.', 'input': 'we went last year ! it never gets old . its rough leaving our collie though', 'response': "it really doesn't . well its good to have someone take care of that .", '

In [50]:
missing = 0

for record in conversation_train:
    if not record["input"].strip() or not record["response"].strip():
        missing += 1

print("Missing records:", missing)

Missing records: 0


In [51]:
from collections import Counter

sources = Counter(record["source"] for record in conversation_train)

print(sources)

Counter({'PersonaChat': 127642, 'DailyDialog': 79157, 'EmpatheticDialogues': 76469})


In [52]:
from collections import Counter

categories = Counter(record["category"] for record in conversation_train)

print(categories)

Counter({'conversation': 283268})


In [53]:
import random

for sample in random.sample(conversation_train, 5):
    print(sample)
    print("=" * 100)

{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': "I can ' t see anything out of the side mirror on your side of the car . Could you move it forward a bit , please ?", 'response': "How ' s that ?", 'category': 'conversation', 'source': 'DailyDialog'}
{'instruction': 'Continue the conversation in a friendly, engaging, and natural way.', 'input': 'how long are you going to be gone ?', 'response': 'i will be gone for 8 days .', 'personality': ['i also like kittens .', 'i work in the military .', 'i have been all over the world .', 'i like things that explode .'], 'category': 'conversation', 'source': 'PersonaChat'}
{'instruction': 'Continue the conversation in a friendly, engaging, and natural way.', 'input': 'i am in college , i am studying to become a dentist', 'response': 'are either of your parents dentists ?', 'personality': ['my adopted dad works at hp .', "i've six siblings .", 'i was adopted when i was a baby .', 'my mom stays at home .'], 'categ

In [54]:
print(f"Total samples: {len(conversation_train):,}")

avg_input = sum(len(r["input"].split()) for r in conversation_train) / len(conversation_train)
avg_response = sum(len(r["response"].split()) for r in conversation_train) / len(conversation_train)

print(f"Average input length: {avg_input:.2f} words")
print(f"Average response length: {avg_response:.2f} words")

Total samples: 283,268
Average input length: 12.86 words
Average response length: 13.53 words


In [57]:
missing = 0

for record in conversation_train:
    if (
        not record["input"].strip()
        or not record["response"].strip()
    ):
        missing += 1

print(f"Missing records: {missing}")

Missing records: 0


In [58]:
import random

for sample in random.sample(conversation_train, 10):
    print(sample)
    print("=" * 100)

{'instruction': 'Respond naturally, warmly, and empathetically to the user.', 'input': "I know. I went to this restaurant one time and when my food arrived I started to eat. That's when I noticed a piece of hair in my sandwich. I was totally grossed out and lost my appetite.", 'response': 'I completely understand. Hopefully the manager was able to take care of it for you!', 'category': 'conversation', 'source': 'EmpatheticDialogues'}
{'instruction': 'Continue the conversation in a friendly, engaging, and natural way.', 'input': 'what kind of shop is it ?', 'response': 'its a clothing shop', 'personality': ["i'm entering the police academy this summer .", 'my prized possession is a bowie knife .', 'i like to watch ma .', 'i lift weights , but i never do squats .', 'i drink protein powder with nothing but water .'], 'category': 'conversation', 'source': 'PersonaChat'}
{'instruction': 'Respond naturally and appropriately in the conversation.', 'input': "Isn't there anything else available

In [59]:
conversation_train = [
    r for r in conversation_train
    if "__SILENCE__" not in r["input"].upper()
    and "__ SILENCE __" not in r["input"].upper()
    and "__SILENCE__" not in r["response"].upper()
    and "__ SILENCE __" not in r["response"].upper()
]

In [60]:
conversation_train = [
    r for r in conversation_train
    if r["input"].strip().lower() != r["response"].strip().lower()
]

In [61]:
from collections import Counter

print("Sources")
print(Counter(r["source"] for r in conversation_train))

print()

print("Categories")
print(Counter(r["category"] for r in conversation_train))

Sources
Counter({'PersonaChat': 122382, 'DailyDialog': 79147, 'EmpatheticDialogues': 71970})

Categories
Counter({'conversation': 273499})


In [62]:
avg_input = sum(len(r["input"].split()) for r in conversation_train)/len(conversation_train)

avg_response = sum(len(r["response"].split()) for r in conversation_train)/len(conversation_train)

print(avg_input)
print(avg_response)

12.984522795330147
13.538978204673509


In [63]:
import json
from collections import Counter

# Basic statistics
total_samples = len(conversation_train)

avg_input_words = sum(
    len(r["input"].split()) for r in conversation_train
) / total_samples

avg_response_words = sum(
    len(r["response"].split()) for r in conversation_train
) / total_samples

source_distribution = dict(
    Counter(r["source"] for r in conversation_train)
)

category_distribution = dict(
    Counter(r["category"] for r in conversation_train)
)

statistics = {
    "dataset": "Conversation Dataset",
    "total_samples": total_samples,
    "average_input_words": round(avg_input_words, 2),
    "average_response_words": round(avg_response_words, 2),
    "sources": source_distribution,
    "categories": category_distribution
}

print(json.dumps(statistics, indent=4))

with open("Conversation/conversation_statistics.json", "w", encoding="utf-8") as f:
    json.dump(statistics, f, indent=4)

print("✅ conversation_statistics.json saved successfully.")

{
    "dataset": "Conversation Dataset",
    "total_samples": 273499,
    "average_input_words": 12.98,
    "average_response_words": 13.54,
    "sources": {
        "PersonaChat": 122382,
        "EmpatheticDialogues": 71970,
        "DailyDialog": 79147
    },
    "categories": {
        "conversation": 273499
    }
}
✅ conversation_statistics.json saved successfully.


In [66]:
with open("Conversation/README.md", "w", encoding="utf-8") as f:
    f.write("# Conversation Dataset\n\n")
    f.write("## Project\n")
    f.write("**Elderly AI Companion System**\n\n")

    f.write("## Module\n")
    f.write("Conversation & Empathy\n\n")

    f.write("## Overview\n")
    f.write("This dataset was prepared for supervised fine-tuning (SFT) of the conversational module of the Elderly AI Companion.\n\n")

    f.write("## Source Datasets\n")
    f.write("- EmpatheticDialogues\n")
    f.write("- PersonaChat\n")
    f.write("- DailyDialog\n\n")

    f.write("## Dataset Format\n")
    f.write("Each sample contains:\n")
    f.write("- instruction\n")
    f.write("- input\n")
    f.write("- response\n")
    f.write("- category\n")
    f.write("- source\n\n")

    f.write("## Preprocessing\n")
    f.write("- Unified all datasets into one JSON format.\n")
    f.write("- Removed duplicate input-response pairs.\n")
    f.write("- Removed empty records.\n")
    f.write("- Removed speaker labels from DailyDialog.\n")
    f.write("- Shuffled the final dataset.\n\n")

    f.write("## Files\n")
    f.write("- empathetic_train.json\n")
    f.write("- persona_train.json\n")
    f.write("- dailydialog_train.json\n")
    f.write("- conversation_train.json\n")
    f.write("- conversation_statistics.json\n")
    f.write("- README.md\n\n")

    f.write("## Intended Use\n")
    f.write("This dataset is intended for supervised fine-tuning of the conversational component of the Elderly AI Companion.\n")

print("README.md created successfully!")

README.md created successfully!


In [67]:
with open("Conversation/README.md", "r", encoding="utf-8") as f:
    print(f.read())

# Conversation Dataset

## Project
**Elderly AI Companion System**

## Module
Conversation & Empathy

## Overview
This dataset was prepared for supervised fine-tuning (SFT) of the conversational module of the Elderly AI Companion.

## Source Datasets
- EmpatheticDialogues
- PersonaChat
- DailyDialog

## Dataset Format
Each sample contains:
- instruction
- input
- response
- category
- source

## Preprocessing
- Unified all datasets into one JSON format.
- Removed duplicate input-response pairs.
- Removed empty records.
- Removed speaker labels from DailyDialog.
- Shuffled the final dataset.

## Files
- empathetic_train.json
- persona_train.json
- dailydialog_train.json
- conversation_train.json
- conversation_statistics.json
- README.md

## Intended Use
This dataset is intended for supervised fine-tuning of the conversational component of the Elderly AI Companion.

